#### Data Prep-02: PIE_vehicle_val_all.csv

In [1]:
# library imports
import os
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
from tqdm import tqdm
import numpy as np

In [2]:
# Configuration
dataset_dir = r"C:\Users\dthq657\OneDrive - University of Leeds\Research\Datasets\PIE"
annotations_dir = os.path.join(dataset_dir, "annotations_vehicle", "set03")
input_csv = r"./annot-data/PIE_annot_attrb_val_fn.csv"
output_csv = r"./obd-data/PIE_vehicle_val_fn.csv"
num_frames = 32  # number of frames prior to critical point

# Load FN subset attributes
df_attr = pd.read_csv(input_csv)
print(f"Loaded subset with {len(df_attr)} pedestrian instances.")

# Extract OBD speeds for each pedestrian
def extract_obd_sequence(ped_id, critical_point):
    """Extracts 32-frame OBD_speed sequence ending at critical_point from vehicle annotations."""
    try:
        set_id, video_id, _ = ped_id.split("_")
        video_filename = f"video_{int(video_id):04d}_obd.xml"
        xml_path = os.path.join(annotations_dir, video_filename)
        if not os.path.exists(xml_path):
            return None
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # Parse all OBD_speed values by frame
        frames = []
        for f in root.findall("frame"):
            frame_id = int(f.attrib["id"])
            obd_speed = float(f.attrib.get("OBD_speed", 0.0))
            frames.append({"frame": frame_id, "OBD_speed": obd_speed})
        df_frames = pd.DataFrame(frames).sort_values("frame").reset_index(drop=True)
        # Select 32 frames ending at the critical point
        seq = df_frames[df_frames["frame"] <= critical_point].tail(num_frames)

        if len(seq) < num_frames:
            # Pad missing frames at the beginning with first value
            pad_len = num_frames - len(seq)
            first_row = seq.iloc[0] if not seq.empty else {"frame": critical_point, "OBD_speed": 0.0}
            pad_df = pd.DataFrame([first_row] * pad_len)
            seq = pd.concat([pad_df, seq], ignore_index=True)
        seq = seq.reset_index(drop=True)

        # Build output dictionary
        row = {"ped_id": ped_id}
        for i, val in enumerate(seq["OBD_speed"], start=1):
            row[f"OBD_{i:02d}"] = val

        # Compute motion classification
        speeds = seq["OBD_speed"].to_numpy()
        diffs = np.diff(speeds)
        mean_speed = np.mean(speeds)
        mean_change = np.mean(diffs)
        abs_change = np.mean(np.abs(diffs))

        # Motion classification (4 categories only)
        # Define thresholds (tunable depending on frame rate & speed range)
        stationary_speed_th = 0.5     # vehicle is nearly stopped
        accel_th = 0.15               # average acceleration threshold
        decel_th = -0.15              # average deceleration threshold
        stable_change_th = 0.05       # for constant-speed detection

        if mean_speed < stationary_speed_th:
            motion = "stationary"
        elif mean_change >= accel_th:
            motion = "accelerating"
        elif mean_change <= decel_th:
            motion = "decelerating"
        elif abs_change < stable_change_th:
            motion = "constant"
        else:
            # For ambiguous or variable cases, assign to the nearest
            motion = "constant" if abs(mean_change) < 0.1 else (
                "accelerating" if mean_change > 0 else "decelerating"
            )
        row["motion_state"] = motion
        return row
    except Exception as e:
        print(f"⚠️ Error processing {ped_id}: {e}")
        return None

# Process all FN pedestrians
records = []
for _, row in tqdm(df_attr.iterrows(), total=len(df_attr), desc="Extracting OBD sequences"):
    ped_id = row["id"]
    critical_point = int(row["critical_point"])
    obd_row = extract_obd_sequence(ped_id, critical_point)
    if obd_row:
        records.append(obd_row)

# Combine results
df_obd_fn = pd.DataFrame(records)

# Save Output
df_obd_fn.to_csv(output_csv, index=False)
print(f"\n✅ Saved vehicle OBD subset with motion states to: {output_csv}")
print(f"📊 Shape: {df_obd_fn.shape}")
df_obd_fn.head()

Loaded subset with 30 pedestrian instances.


Extracting OBD sequences: 100%|██████████| 30/30 [00:04<00:00,  6.56it/s]


✅ Saved vehicle OBD subset with motion states to: ./obd-data/PIE_vehicle_val_fn.csv
📊 Shape: (30, 34)


,ped_id,OBD_01,OBD_02,OBD_03,OBD_04,OBD_05,OBD_06,OBD_07,OBD_08,OBD_09,...,OBD_24,OBD_25,OBD_26,OBD_27,OBD_28,OBD_29,OBD_30,OBD_31,OBD_32,motion_state
0,3_2_290,20.004146,20.004146,20.004146,20.004146,20.004146,20.004146,20.004146,20.004146,20.004146,...,20.004146,20.004146,20.004146,20.004146,18.008559,18.008559,18.008559,18.008559,18.008559,constant
1,3_2_302,15.996879,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,...,14.001293,13.003500,12.005706,12.005706,12.005706,12.005706,12.005706,12.005706,12.005706,decelerating
2,3_2_303,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,decelerating
3,3_3_327,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,...,10.010120,10.010120,10.010120,10.010120,7.998440,7.998440,7.998440,7.000646,6.002853,decelerating
4,3_3_326,19.006353,18.008559,18.008559,18.008559,18.008559,18.008559,18.008559,18.008559,18.008559,...,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,decelerating


In [3]:
# Configuration
dataset_dir = r"C:\Users\dthq657\OneDrive - University of Leeds\Research\Datasets\PIE"
annotations_dir = os.path.join(dataset_dir, "annotations_vehicle", "set03")
input_csv = r"./annot-data/PIE_annot_attrb_val_fp.csv"
output_csv = r"./obd-data/PIE_vehicle_val_fp.csv"
num_frames = 32  # number of frames prior to critical point

# Load fp subset attributes
df_attr = pd.read_csv(input_csv)
print(f"Loaded subset with {len(df_attr)} pedestrian instances.")

# Extract OBD speeds for each pedestrian
def extract_obd_sequence(ped_id, critical_point):
    """Extracts 32-frame OBD_speed sequence ending at critical_point from vehicle annotations."""
    try:
        set_id, video_id, _ = ped_id.split("_")
        video_filename = f"video_{int(video_id):04d}_obd.xml"
        xml_path = os.path.join(annotations_dir, video_filename)
        if not os.path.exists(xml_path):
            return None
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # Parse all OBD_speed values by frame
        frames = []
        for f in root.findall("frame"):
            frame_id = int(f.attrib["id"])
            obd_speed = float(f.attrib.get("OBD_speed", 0.0))
            frames.append({"frame": frame_id, "OBD_speed": obd_speed})
        df_frames = pd.DataFrame(frames).sort_values("frame").reset_index(drop=True)
        # Select 32 frames ending at the critical point
        seq = df_frames[df_frames["frame"] <= critical_point].tail(num_frames)

        if len(seq) < num_frames:
            # Pad missing frames at the beginning with first value
            pad_len = num_frames - len(seq)
            first_row = seq.iloc[0] if not seq.empty else {"frame": critical_point, "OBD_speed": 0.0}
            pad_df = pd.DataFrame([first_row] * pad_len)
            seq = pd.concat([pad_df, seq], ignore_index=True)
        seq = seq.reset_index(drop=True)

        # Build output dictionary
        row = {"ped_id": ped_id}
        for i, val in enumerate(seq["OBD_speed"], start=1):
            row[f"OBD_{i:02d}"] = val

        # Compute motion classification
        speeds = seq["OBD_speed"].to_numpy()
        diffs = np.diff(speeds)
        mean_speed = np.mean(speeds)
        mean_change = np.mean(diffs)
        abs_change = np.mean(np.abs(diffs))

        # Motion classification (4 categories only)
        # Define thresholds (tunable depending on frame rate & speed range)
        stationary_speed_th = 0.5     # vehicle is nearly stopped
        accel_th = 0.15               # average acceleration threshold
        decel_th = -0.15              # average deceleration threshold
        stable_change_th = 0.05       # for constant-speed detection

        if mean_speed < stationary_speed_th:
            motion = "stationary"
        elif mean_change >= accel_th:
            motion = "accelerating"
        elif mean_change <= decel_th:
            motion = "decelerating"
        elif abs_change < stable_change_th:
            motion = "constant"
        else:
            # For ambiguous or variable cases, assign to the nearest
            motion = "constant" if abs(mean_change) < 0.1 else (
                "accelerating" if mean_change > 0 else "decelerating"
            )
        row["motion_state"] = motion
        return row
    except Exception as e:
        print(f"⚠️ Error processing {ped_id}: {e}")
        return None

# Process all fp pedestrians
records = []
for _, row in tqdm(df_attr.iterrows(), total=len(df_attr), desc="Extracting OBD sequences"):
    ped_id = row["id"]
    critical_point = int(row["critical_point"])
    obd_row = extract_obd_sequence(ped_id, critical_point)
    if obd_row:
        records.append(obd_row)

# Combine results
df_obd_fp = pd.DataFrame(records)

# Save Output
df_obd_fp.to_csv(output_csv, index=False)
print(f"\n✅ Saved vehicle OBD subset with motion states to: {output_csv}")
print(f"📊 Shape: {df_obd_fp.shape}")
df_obd_fp.head()

Loaded subset with 30 pedestrian instances.


Extracting OBD sequences:   0%|          | 0/30 [00:00<?, ?it/s]

Extracting OBD sequences: 100%|██████████| 30/30 [00:04<00:00,  6.93it/s]


✅ Saved vehicle OBD subset with motion states to: ./obd-data/PIE_vehicle_val_fp.csv
📊 Shape: (30, 34)


,ped_id,OBD_01,OBD_02,OBD_03,OBD_04,OBD_05,OBD_06,OBD_07,OBD_08,OBD_09,...,OBD_24,OBD_25,OBD_26,OBD_27,OBD_28,OBD_29,OBD_30,OBD_31,OBD_32,motion_state
0,3_1_266,6.002853,6.002853,6.002853,6.002853,6.002853,6.002853,6.002853,6.002853,6.002853,...,6.002853,6.002853,6.002853,6.002853,6.002853,6.002853,6.002853,6.002853,6.002853,constant
1,3_1_248,18.008559,18.008559,18.008559,18.008559,18.008559,18.008559,18.008559,18.008559,18.008559,...,20.004146,20.004146,20.004146,20.004146,20.004146,20.004146,20.004146,21.999732,21.999732,accelerating
2,3_2_296,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,...,10.010120,10.010120,11.007913,12.005706,12.005706,12.005706,12.005706,12.005706,13.003500,accelerating
3,3_2_295,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,6.002853,...,10.010120,11.007913,12.005706,12.005706,12.005706,12.005706,12.005706,13.003500,14.001293,accelerating
4,3_2_294,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,...,10.010120,10.010120,11.007913,12.005706,12.005706,12.005706,12.005706,12.005706,13.003500,accelerating


In [4]:
# Configuration
dataset_dir = r"C:\Users\dthq657\OneDrive - University of Leeds\Research\Datasets\PIE"
annotations_dir = os.path.join(dataset_dir, "annotations_vehicle", "set03")
input_csv = r"./annot-data/PIE_annot_attrb_val_tp.csv"
output_csv = r"./obd-data/PIE_vehicle_val_tp.csv"
num_frames = 32  # number of frames prior to critical point

# Load tp subset attributes
df_attr = pd.read_csv(input_csv)
print(f"Loaded subset with {len(df_attr)} pedestrian instances.")

# Extract OBD speeds for each pedestrian
def extract_obd_sequence(ped_id, critical_point):
    """Extracts 32-frame OBD_speed sequence ending at critical_point from vehicle annotations."""
    try:
        set_id, video_id, _ = ped_id.split("_")
        video_filename = f"video_{int(video_id):04d}_obd.xml"
        xml_path = os.path.join(annotations_dir, video_filename)
        if not os.path.exists(xml_path):
            return None
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # Parse all OBD_speed values by frame
        frames = []
        for f in root.findall("frame"):
            frame_id = int(f.attrib["id"])
            obd_speed = float(f.attrib.get("OBD_speed", 0.0))
            frames.append({"frame": frame_id, "OBD_speed": obd_speed})
        df_frames = pd.DataFrame(frames).sort_values("frame").reset_index(drop=True)
        # Select 32 frames ending at the critical point
        seq = df_frames[df_frames["frame"] <= critical_point].tail(num_frames)

        if len(seq) < num_frames:
            # Pad missing frames at the beginning with first value
            pad_len = num_frames - len(seq)
            first_row = seq.iloc[0] if not seq.empty else {"frame": critical_point, "OBD_speed": 0.0}
            pad_df = pd.DataFrame([first_row] * pad_len)
            seq = pd.concat([pad_df, seq], ignore_index=True)
        seq = seq.reset_index(drop=True)

        # Build output dictionary
        row = {"ped_id": ped_id}
        for i, val in enumerate(seq["OBD_speed"], start=1):
            row[f"OBD_{i:02d}"] = val

        # Compute motion classification
        speeds = seq["OBD_speed"].to_numpy()
        diffs = np.diff(speeds)
        mean_speed = np.mean(speeds)
        mean_change = np.mean(diffs)
        abs_change = np.mean(np.abs(diffs))

        # Motion classification (4 categories only)
        # Define thresholds (tunable depending on frame rate & speed range)
        stationary_speed_th = 0.5     # vehicle is nearly stopped
        accel_th = 0.15               # average acceleration threshold
        decel_th = -0.15              # average deceleration threshold
        stable_change_th = 0.05       # for constant-speed detection

        if mean_speed < stationary_speed_th:
            motion = "stationary"
        elif mean_change >= accel_th:
            motion = "accelerating"
        elif mean_change <= decel_th:
            motion = "decelerating"
        elif abs_change < stable_change_th:
            motion = "constant"
        else:
            # For ambiguous or variable cases, assign to the nearest
            motion = "constant" if abs(mean_change) < 0.1 else (
                "accelerating" if mean_change > 0 else "decelerating"
            )
        row["motion_state"] = motion
        return row
    except Exception as e:
        print(f"⚠️ Error processing {ped_id}: {e}")
        return None

# Process all tp pedestrians
records = []
for _, row in tqdm(df_attr.iterrows(), total=len(df_attr), desc="Extracting OBD sequences"):
    ped_id = row["id"]
    critical_point = int(row["critical_point"])
    obd_row = extract_obd_sequence(ped_id, critical_point)
    if obd_row:
        records.append(obd_row)

# Combine results
df_obd_tp = pd.DataFrame(records)

# Save Output
df_obd_tp.to_csv(output_csv, index=False)
print(f"\n✅ Saved vehicle OBD subset with motion states to: {output_csv}")
print(f"📊 Shape: {df_obd_tp.shape}")
df_obd_tp.head()

Loaded subset with 177 pedestrian instances.


Extracting OBD sequences:   0%|          | 0/177 [00:00<?, ?it/s]

Extracting OBD sequences: 100%|██████████| 177/177 [00:25<00:00,  7.01it/s]


✅ Saved vehicle OBD subset with motion states to: ./obd-data/PIE_vehicle_val_tp.csv
📊 Shape: (177, 34)


,ped_id,OBD_01,OBD_02,OBD_03,OBD_04,OBD_05,OBD_06,OBD_07,OBD_08,OBD_09,...,OBD_24,OBD_25,OBD_26,OBD_27,OBD_28,OBD_29,OBD_30,OBD_31,OBD_32,motion_state
0,3_1_267,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,stationary
1,3_1_268,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,stationary
2,3_2_289,18.008559,18.008559,17.002719,15.996879,15.996879,15.996879,15.996879,15.996879,15.996879,...,14.999086,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,decelerating
3,3_2_304,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,stationary
4,3_2_305,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,stationary


In [5]:
# Configuration
dataset_dir = r"C:\Users\dthq657\OneDrive - University of Leeds\Research\Datasets\PIE"
annotations_dir = os.path.join(dataset_dir, "annotations_vehicle", "set03")
input_csv = r"./annot-data/PIE_annot_attrb_val_tn.csv"
output_csv = r"./obd-data/PIE_vehicle_val_tn.csv"
num_frames = 32  # number of frames prior to critical point

# Load tn subset attributes
df_attr = pd.read_csv(input_csv)
print(f"Loaded subset with {len(df_attr)} pedestrian instances.")

# Extract OBD speeds for each pedestrian
def extract_obd_sequence(ped_id, critical_point):
    """Extracts 32-frame OBD_speed sequence ending at critical_point from vehicle annotations."""
    try:
        set_id, video_id, _ = ped_id.split("_")
        video_filename = f"video_{int(video_id):04d}_obd.xml"
        xml_path = os.path.join(annotations_dir, video_filename)
        if not os.path.exists(xml_path):
            return None
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # Parse all OBD_speed values by frame
        frames = []
        for f in root.findall("frame"):
            frame_id = int(f.attrib["id"])
            obd_speed = float(f.attrib.get("OBD_speed", 0.0))
            frames.append({"frame": frame_id, "OBD_speed": obd_speed})
        df_frames = pd.DataFrame(frames).sort_values("frame").reset_index(drop=True)
        # Select 32 frames ending at the critical point
        seq = df_frames[df_frames["frame"] <= critical_point].tail(num_frames)

        if len(seq) < num_frames:
            # Pad missing frames at the beginning with first value
            pad_len = num_frames - len(seq)
            first_row = seq.iloc[0] if not seq.empty else {"frame": critical_point, "OBD_speed": 0.0}
            pad_df = pd.DataFrame([first_row] * pad_len)
            seq = pd.concat([pad_df, seq], ignore_index=True)
        seq = seq.reset_index(drop=True)

        # Build output dictionary
        row = {"ped_id": ped_id}
        for i, val in enumerate(seq["OBD_speed"], start=1):
            row[f"OBD_{i:02d}"] = val

        # Compute motion classification
        speeds = seq["OBD_speed"].to_numpy()
        diffs = np.diff(speeds)
        mean_speed = np.mean(speeds)
        mean_change = np.mean(diffs)
        abs_change = np.mean(np.abs(diffs))

        # Motion classification (4 categories only)
        # Define thresholds (tunable depending on frame rate & speed range)
        stationary_speed_th = 0.5     # vehicle is nearly stopped
        accel_th = 0.15               # average acceleration threshold
        decel_th = -0.15              # average deceleration threshold
        stable_change_th = 0.05       # for constant-speed detection

        if mean_speed < stationary_speed_th:
            motion = "stationary"
        elif mean_change >= accel_th:
            motion = "accelerating"
        elif mean_change <= decel_th:
            motion = "decelerating"
        elif abs_change < stable_change_th:
            motion = "constant"
        else:
            # For ambiguous or variable cases, assign to the nearest
            motion = "constant" if abs(mean_change) < 0.1 else (
                "accelerating" if mean_change > 0 else "decelerating"
            )
        row["motion_state"] = motion
        return row
    except Exception as e:
        print(f"⚠️ Error processing {ped_id}: {e}")
        return None

# Process all tn pedestrians
records = []
for _, row in tqdm(df_attr.iterrows(), total=len(df_attr), desc="Extracting OBD sequences"):
    ped_id = row["id"]
    critical_point = int(row["critical_point"])
    obd_row = extract_obd_sequence(ped_id, critical_point)
    if obd_row:
        records.append(obd_row)

# Combine results
df_obd_tn = pd.DataFrame(records)

# Save Output
df_obd_tn.to_csv(output_csv, index=False)
print(f"\n✅ Saved vehicle OBD subset with motion states to: {output_csv}")
print(f"📊 Shape: {df_obd_tn.shape}")
df_obd_tn.head()

Loaded subset with 482 pedestrian instances.


Extracting OBD sequences:   0%|          | 0/482 [00:00<?, ?it/s]

Extracting OBD sequences: 100%|██████████| 482/482 [01:09<00:00,  6.92it/s]


✅ Saved vehicle OBD subset with motion states to: ./obd-data/PIE_vehicle_val_tn.csv
📊 Shape: (482, 34)


,ped_id,OBD_01,OBD_02,OBD_03,OBD_04,OBD_05,OBD_06,OBD_07,OBD_08,OBD_09,...,OBD_24,OBD_25,OBD_26,OBD_27,OBD_28,OBD_29,OBD_30,OBD_31,OBD_32,motion_state
0,3_1_232,36.001025,36.001025,36.001025,36.001025,36.001025,36.001025,36.001025,36.001025,36.001025,...,36.001025,36.001025,36.001025,36.001025,36.001025,36.001025,36.001025,36.001025,36.001025,constant
1,3_1_237,10.010120,10.010120,12.005706,12.005706,12.005706,12.005706,12.005706,12.005706,12.005706,...,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,accelerating
2,3_1_255,26.006999,26.006999,26.006999,26.006999,26.006999,26.006999,26.006999,26.006999,26.006999,...,28.002586,28.002586,28.002586,28.002586,28.002586,28.002586,28.002586,28.002586,28.002586,constant
3,3_1_231,36.001025,36.001025,36.001025,36.001025,36.001025,36.001025,36.001025,36.001025,36.001025,...,36.001025,36.001025,36.001025,36.001025,36.001025,36.001025,36.001025,36.001025,36.001025,constant
4,3_1_224,30.014266,29.008426,28.002586,28.002586,28.002586,28.002586,28.002586,28.002586,28.002586,...,28.002586,28.002586,28.002586,28.002586,28.002586,28.002586,28.002586,28.002586,28.002586,constant


In [9]:
# Paths
base_dir = "./obd-data"
file_map = {
    "FN": os.path.join(base_dir, "PIE_vehicle_val_fn.csv"),
    "FP": os.path.join(base_dir, "PIE_vehicle_val_fp.csv"),
    "TP": os.path.join(base_dir, "PIE_vehicle_val_tp.csv"),
    "TN": os.path.join(base_dir, "PIE_vehicle_val_tn.csv"),
}

# Combine into one dataframe
dfs = []
for label, path in file_map.items():
    df = pd.read_csv(path)
    df["subset"] = label  # add subset identifier
    dfs.append(df)

# Concatenate all
df_all = pd.concat(dfs, ignore_index=True)

# Sort by subset or ped_id
df_all = df_all.sort_values(by=["subset", "ped_id"]).reset_index(drop=True)

# Save to CSV
output_path = os.path.join('./', "PIE_val_vehicle.csv")
df_all.to_csv(output_path, index=False)

print(f"✅ Combined file saved to: {output_path}")
print(f"📊 Shape: {df_all.shape}")
df_all.head(10)

✅ Combined file saved to: ./PIE_val_vehicle.csv
📊 Shape: (719, 35)


,ped_id,OBD_01,OBD_02,OBD_03,OBD_04,OBD_05,OBD_06,OBD_07,OBD_08,OBD_09,...,OBD_25,OBD_26,OBD_27,OBD_28,OBD_29,OBD_30,OBD_31,OBD_32,motion_state,subset
0,3_2_290,20.004146,20.004146,20.004146,20.004146,20.004146,20.004146,20.004146,20.004146,20.004146,...,20.004146,20.004146,20.004146,18.008559,18.008559,18.008559,18.008559,18.008559,constant,FN
1,3_2_302,15.996879,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,...,13.003500,12.005706,12.005706,12.005706,12.005706,12.005706,12.005706,12.005706,decelerating,FN
2,3_2_303,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,decelerating,FN
3,3_3_309,28.002586,28.002586,28.002586,28.002586,28.002586,28.002586,28.002586,28.002586,28.002586,...,28.002586,28.002586,27.004792,26.006999,26.006999,26.006999,26.006999,26.006999,constant,FN
4,3_3_326,19.006353,18.008559,18.008559,18.008559,18.008559,18.008559,18.008559,18.008559,18.008559,...,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,decelerating,FN
5,3_3_327,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,...,10.010120,10.010120,10.010120,7.998440,7.998440,7.998440,7.000646,6.002853,decelerating,FN
6,3_3_337,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,stationary,FN
7,3_3_341,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,4.007267,...,6.002853,6.002853,6.002853,6.002853,6.002853,6.002853,6.002853,6.002853,constant,FN
8,3_4_351,18.008559,18.008559,18.008559,18.008559,18.008559,15.996879,15.996879,15.996879,15.996879,...,14.001293,14.001293,14.001293,14.001293,14.001293,14.001293,13.003500,12.005706,decelerating,FN
9,3_4_356,7.998440,7.998440,7.998440,7.998440,7.998440,7.998440,7.998440,7.998440,7.998440,...,6.002853,6.002853,6.002853,6.002853,6.002853,6.002853,6.002853,6.002853,constant,FN


In [8]:
df_all['subset'].value_counts()

subset
TN    482
TP    177
FP     30
FN     30
Name: count, dtype: int64